# The Price Is Right

### Evaluate our fine-tuned **open-source** model (Llama-3.2-3B + QLoRA/PEFT)

This notebook loads only free/open-source models from Hugging Face. No proprietary/paid APIs or closed models are used.

**Cell purpose:** Install/upgrade the required libraries (`bitsandbytes` for 4-bit quantization and `trl`) and download the helper `util.py` that contains the `evaluate` function used later.

In [1]:
!pip install -q --upgrade bitsandbytes trl
!wget -q https://raw.githubusercontent.com/Abhishekravindran/LLM_Engineering_opensource/main/finetuning_local_frontier_models/util.py -O util.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.4 MB/s eta 0:00:00


**Cell purpose:** Import all Python packages needed for the rest of the notebook (PyTorch, Transformers, PEFT, datasets, Hugging Face Hub login, and the custom `evaluate` helper).

In [2]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

**Cell purpose:** Define all constants — base open-source model name, project/run identifiers, Hugging Face dataset name, and whether to use 4-bit quantization. Also detect GPU capability to choose bfloat16 vs float16.

In [22]:
# Constants — your Qwen fine-tuned model

BASE_MODEL = "Qwen/Qwen3-0.6B"          # ← MUST match adapter_config.json
PROJECT_NAME = "price"

HF_USER = "Charan2804"
RUN_NAME = "2026-08-25_08.25.57-lite"   # change if you use a different repo
REVISION = None

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

LITE_MODE = True
DATA_USER = "ed-donner"                 # public dataset is fine
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

print(f"Will load adapter from: {HUB_MODEL_NAME}")
print(f"Base model: {BASE_MODEL}")
print(f"Dataset: {DATASET_NAME}")

Will load adapter from: Charan2804/price-2026-08-25_08.25.57-lite
Base model: Qwen/Qwen3-0.6B
Dataset: ed-donner/items_prompts_lite


### Log in to Hugging Face

If you don't already have a Hugging Face account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon on the left, and add a new secret called `HF_TOKEN` with the value of your token.

**Cell purpose:** Retrieve the Hugging Face token stored in Colab secrets and log in so we can download the gated Llama base model and the fine-tuned adapter.

In [25]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

**Cell purpose:** Load the evaluation dataset (test split) from Hugging Face. This contains the product prompts we will use to measure price-prediction performance.

In [26]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

**Cell purpose:** Inspect the first example in the test set so we can see the structure of a prompt and the expected price.

In [27]:
test[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $',
 'completion': '219.0'}

## Load the Tokenizer and the fine-tuned open-source Model

**Cell purpose:** Configure BitsAndBytes quantization (4-bit NF4 by default, or 8-bit as fallback) so the model fits comfortably on a free T4 GPU.

In [28]:
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

**Cell purpose:** Load the Llama-3.2-3B tokenizer and base model with quantization, then attach the fine-tuned PEFT/LoRA adapter from the Hugging Face Hub. Prints the memory footprint of the resulting model.

In [29]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned PEFT adapter (open-source only)
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Memory footprint: 571.9 MB


**Cell purpose:** Display a summary of the loaded fine-tuned model architecture and adapters for a quick sanity check.

In [30]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

# Inference & Evaluation

Run the fine-tuned open-source model on the held-out test set.

**Caveat:** Product prices vary (sales, promotions, region). The model can only predict from the information present in the prompt.

**Cell purpose:** Define a simple prediction function that tokenizes a product prompt, generates a short continuation with the fine-tuned model, and returns only the newly generated tokens (the predicted price).

In [31]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

**Cell purpose:** Set a fixed random seed for reproducibility and call the `evaluate` helper on the entire test set. This reports average absolute error (and related metrics) for the open-source fine-tuned model.

In [32]:
set_seed(42)
evaluate(model_predict, test)